# Waste Segregation — GPU Training (Google Colab)

**Before opening this notebook, on your laptop (PowerShell, project root):**
```powershell
Compress-Archive -Path data\processed -DestinationPath processed_colab.zip -Force
```
You will upload 4 files in Step 2: `processed_colab.zip`, `src/train.py`,
`src/preprocessing.py`, `models/class_names.json`.

Runtime used here: Colab T4 GPU. Training takes ~10-25 min total.
After training, download the `.keras` model back and continue locally at Phase 6.

## Step 1 — Check GPU and TensorFlow version

In [ ]:
import tensorflow as tf
print("TF:", tf.__version__)
print("Keras:", tf.keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
assert gpus, "No GPU! Go to Runtime > Change runtime type > T4 GPU."

## Step 2 — Upload project files

Upload these 4 files when prompted: `processed_colab.zip`, `train.py`,
`preprocessing.py`, `class_names.json`.

In [ ]:
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

## Step 3 — Unzip dataset and verify layout

In [ ]:
import pathlib
!unzip -q -o processed_colab.zip
for split in ("train", "val", "test"):
    for cls in ("metal", "organic", "paper", "plastic"):
        n = len(list(pathlib.Path(f"processed/{split}/{cls}").glob("*.*")))
        print(f"{split}/{cls} = {n}")
print("train.py present:", pathlib.Path("train.py").exists())
print("preprocessing.py present:", pathlib.Path("preprocessing.py").exists())

Expected counts: train metal 287 / organic 349 / paper 697 / plastic 337,
val 61 / 74 / 149 / 72, test 62 / 76 / 151 / 73 (total 2,388).
If these differ, STOP — the zip is wrong.

## Step 4 — Train (Stage A frozen, then Stage B fine-tune)

In [ ]:
!python train.py --data-dir processed --models-dir models_colab \
    --history-dir hist_colab --class-names-json class_names.json \
    --epochs-a 25 --epochs-b 20 --batch-size 32

## Step 5 — Verify outputs and download them back

Run the check cell first: all 4 files must show OK (if one is MISSING,
training did not finish — re-run Step 4). Then run the zip cell: it packs
everything into ONE `waste_outputs.zip` so you download a single file.

Back on your laptop, unzip and place:
`waste_classifier.keras` + `stage_a_best.keras` → `models/`;
`history_stage_a.json` + `history_stage_b.json` → `outputs/`.
Then continue locally with Phase 6 (`evaluate.py`).

In [ ]:
import pathlib
wanted = [
    "models_colab/waste_classifier.keras",
    "models_colab/stage_a_best.keras",
    "hist_colab/history_stage_a.json",
    "hist_colab/history_stage_b.json",
]
ok = True
for f in wanted:
    p = pathlib.Path(f)
    if p.is_file():
        print(f"OK       {p.stat().st_size/1e6:5.1f} MB  {f}")
    else:
        ok = False
        print(f"MISSING  {f}")
assert ok, "A file is MISSING — did Step 4 finish? Re-run it."

In [ ]:
!zip -q -r waste_outputs.zip models_colab hist_colab
import pathlib
from google.colab import files as _dl
p = pathlib.Path("waste_outputs.zip")
print(f"waste_outputs.zip: {p.stat().st_size/1e6:.1f} MB")
_dl.download("waste_outputs.zip")

## Step 6 (optional but recommended) — Back up to Google Drive

Colab disks are wiped on reconnect. This copies the model + histories to
your Drive, where they survive. Run it right after training, BEFORE any
reconnect.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/waste_backup"
!cp models_colab/waste_classifier.keras models_colab/stage_a_best.keras "/content/drive/MyDrive/waste_backup/"
!cp hist_colab/history_stage_a.json hist_colab/history_stage_b.json "/content/drive/MyDrive/waste_backup/"
!ls -la "/content/drive/MyDrive/waste_backup/"